In [4]:
pip install protobuf==3.20.3


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mlflow
import mlflow.keras   # or mlflow.pytorch if you use PyTorch

# Replace with your actual DagsHub username and repo name
import dagshub
dagshub.init(
    repo_owner="ahmed.adel1711",
    repo_name="SLR-Main",
    mlflow=True
)
# This one line connects MLflow to the cloud. Nothing stored locally.

Accessing as hadeelgamal.sis.2020

Initialized MLflow to track repo "ahmed.adel1711/SLR-Main"

Repository ahmed.adel1711/SLR-Main initialized!

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical  # type: ignore
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau  # type: ignore

# ── Reproduce the exact dataset / splits used in Mediapipe_Training.ipynb ──
CSV_PATH = "asl_letters_merged.csv"   # actual dataset file used in training
df = pd.read_csv(CSV_PATH)

X = df.iloc[:, :-1].astype("float32").values
y = df["label"].values

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)   # 28 for asl_letters_merged.csv

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

X_train = X_train.astype("float32")
X_val   = X_val.astype("float32")
X_test  = X_test.astype("float32")

y_train = to_categorical(y_train, num_classes=num_classes)
y_val   = to_categorical(y_val,   num_classes=num_classes)
y_test  = to_categorical(y_test,  num_classes=num_classes)

# ── GPU / device detection (mirrors training notebook) ──────────────────────
gpus = tf.config.list_physical_devices('GPU')
USE_GPU = len(gpus) > 0
DEVICE  = '/GPU:0' if USE_GPU else '/CPU:0'
BATCH_SIZE      = 256 if USE_GPU else 64
eval_batch_size = 256 if USE_GPU else 128

# ── tf.data pipelines ───────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(features, labels, batch_size, training=True):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(len(features), 10000), reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, BATCH_SIZE, training=True)
val_ds   = make_dataset(X_val,   y_val,   BATCH_SIZE, training=False)
test_ds  = make_dataset(X_test,  y_test,  eval_batch_size, training=False)

# ── Callbacks (same as training notebook) ───────────────────────────────────
callbacks = [
    ModelCheckpoint(
        'asl_mediapipe_mlp_model.h5',
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    )
]

# ── Load the trained model ───────────────────────────────────────────────────
# ModelCheckpoint saves the best weights to 'asl_mediapipe_mlp_model.h5'
model = load_model("asl_mediapipe_mlp_model.h5")

with mlflow.start_run(run_name="MediaPipe_training"):

    # --- LOG ARCHITECTURE & MODEL CONFIG ---
    mlflow.log_param("architecture", "MLP")
    mlflow.log_param("landmark_type", "mediapipe_hands")
    mlflow.log_param("input_features", 63)          # 21 landmarks × (x, y, z)
    mlflow.log_param("output_classes", num_classes)  # dynamic from dataset (28)

    # Actual architecture from Mediapipe_Training.ipynb
    mlflow.log_param("layers", "256→128→64→output")
    mlflow.log_param("activation", "relu")
    mlflow.log_param("output_activation", "softmax")
    mlflow.log_param("kernel_initializer", "he_normal")
    mlflow.log_param("l2_regularization", 1e-4)     # applied to first 2 Dense layers
    mlflow.log_param("dropout_rates", "0.3, 0.25, 0.2")
    mlflow.log_param("batch_normalization", True)

    # --- LOG TRAINING CONFIG ---
    lr_value = (float(model.optimizer.learning_rate.numpy())
                if hasattr(model.optimizer.learning_rate, 'numpy')
                else float(model.optimizer.learning_rate))
    mlflow.log_param("learning_rate", lr_value)
    mlflow.log_param("optimizer", model.optimizer.__class__.__name__)  # Adam (legacy)
    mlflow.log_param("loss", "categorical_crossentropy")
    mlflow.log_param("batch_size", BATCH_SIZE)           # 256 (GPU) / 64 (CPU)
    mlflow.log_param("epochs_max", 20)
    mlflow.log_param("early_stopping_patience", 5)
    mlflow.log_param("reduce_lr_factor", 0.5)
    mlflow.log_param("reduce_lr_patience", 3)
    mlflow.log_param("min_lr", 1e-7)
    mlflow.log_param("mixed_precision", USE_GPU)

    # --- LOG DATA CONFIG ---
    mlflow.log_param("dataset", CSV_PATH)              # asl_letters_merged.csv
    mlflow.log_param("train_samples", len(X_train))
    mlflow.log_param("val_samples", len(X_val))
    mlflow.log_param("test_samples_param", len(X_test))
    mlflow.log_param("train_split", 0.64)
    mlflow.log_param("val_split", 0.16)
    mlflow.log_param("test_split", 0.20)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("stratified_split", True)

    # --- TRAIN ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=callbacks,
        verbose=1
    )

    # --- LOG METRICS PER EPOCH ---
    for epoch, (loss, acc, val_loss, val_acc) in enumerate(zip(
            history.history['loss'],
            history.history['accuracy'],
            history.history['val_loss'],
            history.history['val_accuracy'])):
        mlflow.log_metric("train_loss", loss, step=epoch)
        mlflow.log_metric("train_accuracy", acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)

    # --- TEST EVALUATION ---
    with tf.device(DEVICE):
        test_loss, test_acc = model.evaluate(test_ds, verbose=1)
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_samples", len(X_test))

    # --- ARTIFACTS ---
    # ModelCheckpoint saves to 'asl_mediapipe_mlp_model.h5' (best val_accuracy)
    mlflow.log_artifact("asl_mediapipe_mlp_model.h5")

    # --- LOG MODEL ---
    mlflow.keras.log_model(model, "model")


Epoch 1/20
155/156 [============================>.] - ETA: 0s - loss: 0.1589 - accuracy: 0.9663
Epoch 1: val_accuracy improved from -inf to 0.98196, saving model to asl_mediapipe_mlp_model.h5
156/156 [==============================] - 5s 11ms/step - loss: 0.1590 - accuracy: 0.9663 - val_loss: 0.1073 - val_accuracy: 0.9820 - lr: 5.0000e-04
Epoch 2/20
154/156 [============================>.] - ETA: 0s - loss: 0.1396 - accuracy: 0.9726
Epoch 2: val_accuracy improved from 0.98196 to 0.98659, saving model to asl_mediapipe_mlp_model.h5
156/156 [==============================] - 2s 10ms/step - loss: 0.1401 - accuracy: 0.9725 - val_loss: 0.0970 - val_accuracy: 0.9866 - lr: 5.0000e-04
Epoch 3/20
156/156 [==============================] - ETA: 0s - loss: 0.1337 - accuracy: 0.9741
Epoch 3: val_accuracy did not improve from 0.98659
156/156 [==============================] - 1s 9ms/step - loss: 0.1337 - accuracy: 0.9741 - val_loss: 0.0972 - val_accuracy: 0.9865 - lr: 5.0000e-04
Epoch 4/20
150/156 [

2026/05/06 12:15:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 12:16:09 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\HADEEL~1\AppData\Local\Temp\tmpmngzov7t\model\data\model\assets


2026/05/06 12:18:23 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\HADEEL~1\AppData\Local\Temp\tmpmngzov7t\model, flavor: tensorflow). Fall back to return ['tensorflow==2.10.0']. Set logging level to DEBUG to see the full traceback. 


🏃 View run MediaPipe_training at: https://dagshub.com/ahmed.adel1711/SLR-Main.mlflow/#/experiments/0/runs/8a67f6ccfa0c49e3bfe56de1e94deb68
🧪 View experiment at: https://dagshub.com/ahmed.adel1711/SLR-Main.mlflow/#/experiments/0


Step 4 - register the best model 

In [ ]:
# After comparing runs in the UI, register your winner
# Do this ONCE manually in the DagsHub UI:
# Click the run → "Register Model" → name it "SLR-Production"

# Or do it in code:
#run_id = "paste-the-run-id-from-the-ui"
#mlflow.register_model(
#    model_uri=f"runs:/{run_id}/model",
#    name="SLR-Production"
#)


# step 5 - clean your repo
# Add to .gitignore — never commit local MLflow data
mlruns/
mlflow.db
*.h5          # old model files
*.pkl         # old pickle files
confusion_matrix.png  # now stored in MLflow, not needed here